# Тестовое задание: расчёты

In [1]:
import glob
import numpy as np
import pandas as pd
from scipy import stats

data_file = glob.glob('Данные*тестов*.xlsx')[0]
audience = pd.read_excel(data_file, sheet_name='Данные об аудитории')
ab = pd.read_excel(data_file, sheet_name='Данные АБ тестов')
listers = pd.read_excel(data_file, sheet_name='Листеры')
audience.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


### 1. MAU

In [2]:
mau = audience['user_id'].nunique()
mau

7639

### 2. DAU

In [3]:
dau = audience.groupby('date')['user_id'].nunique()
dau.mean()

np.float64(560.4666666666667)

### 3. Retention первого дня для пользователей, пришедших 1 ноября

In [4]:
first_day = audience.groupby('user_id')['date'].min()
cohort = first_day[first_day == '2023-11-01'].index
returned = audience[(audience['user_id'].isin(cohort)) & (audience['date'] == '2023-11-02')]['user_id'].nunique()
print(len(cohort), returned)
returned / len(cohort)

623 166


0.2664526484751204

### 5. Пользовательская конверсия в просмотр объявления

In [5]:
views_per_user = audience.groupby('user_id')['view_adverts'].sum()
(views_per_user > 0).mean()

np.float64(0.4631496269145176)

### 6. Среднее количество просмотренных объявлений на пользователя

In [6]:
views_per_user.mean()

np.float64(2.8687000916350307)

### 7. NPS

In [7]:
promoters, detractors, total = 1200, 500, 2000
(promoters / total - detractors / total) * 100

35.0

### 8. АБ-тесты (ARPU)

In [8]:
rows = []
for exp, group in ab.groupby('experiment_num'):
    test = group[group['experiment_group'] == 'test']['revenue']
    control = group[group['experiment_group'] == 'control']['revenue']
    rows.append({
        'эксперимент': exp,
        'ARPU test': round(test.mean(), 1),
        'ARPU control': round(control.mean(), 1),
        'изменение, %': round((test.mean() / control.mean() - 1) * 100, 1),
        'p-value t-тест': round(stats.ttest_ind(test, control, equal_var=False).pvalue, 4),
        'p-value Манн-Уитни': round(stats.mannwhitneyu(test, control).pvalue, 4),
    })
pd.DataFrame(rows)

,эксперимент,ARPU test,ARPU control,"изменение, %",p-value t-тест,p-value Манн-Уитни
0,1,665.7,722.5,-7.9,0.6890,0.7960
1,2,332.9,704.7,-52.8,0.0011,0.0085
2,3,998.7,663.2,50.6,0.0603,0.0010


In [9]:
# бутстрап доверительного интервала разницы ARPU (выручка с большими выбросами)
rng = np.random.default_rng(0)
for exp, group in ab.groupby('experiment_num'):
    test = group[group['experiment_group'] == 'test']['revenue'].values
    control = group[group['experiment_group'] == 'control']['revenue'].values
    diffs = [rng.choice(test, len(test)).mean() - rng.choice(control, len(control)).mean() for _ in range(10000)]
    print(exp, np.percentile(diffs, [2.5, 97.5]).round(1))

1 [-335.2  224.6]
2 [-605.7 -164.7]
3 [ -3.8 693.3]


### 9. Средний доход на пользователя (листеры)

In [10]:
listers.groupby('user_id')['revenue'].sum().mean()

np.float64(156.48387096774192)

### 10. Медиана возраста пользователя

In [11]:
listers.groupby('user_id')['age'].first().median()

np.float64(28.0)

### 18. Проверка результатов эксперимента с платежами

In [12]:
n_a, x_a = 100_047_501, 1003
n_b, x_b = 100_001_055, 1099

p_a, p_b = x_a / n_a, x_b / n_b
p_pool = (x_a + x_b) / (n_a + n_b)
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
z = (p_b - p_a) / se
p_value = 2 * stats.norm.sf(abs(z))
print(f'конверсия A = {p_a:.6%}, B = {p_b:.6%}, рост = {(p_b / p_a - 1):.1%}')
print(f'z = {z:.2f}, p-value = {p_value:.4f}')

# проверка равномерности сплита (SRM)
print('SRM p-value:', stats.chisquare([n_a, n_b]).pvalue)

конверсия A = 0.001003%, B = 0.001099%, рост = 9.6%
z = 2.10, p-value = 0.0353
SRM p-value: 0.0010240683582667417
